# System-One & System-Two Decision Server (SGLang + Docling)
### Turnkey Google Colab Backend for ARC CUA & Local System-1 Reflex Architectures

---

### Operational Constraints & Compute Economics (Google AI Pro)
- **Monthly Budget**: 200 Colab Compute Units (CCUs)/month.
- **GPU Selection Strategy**:
  - **L4 GPU (24GB VRAM)**: ~4–6 CCUs/hr (~35–50 hours/month). Default for routine decision routing, administrative triage, and daemon workloads.
  - **A100 GPU (40GB VRAM)**: ~12–15 CCUs/hr (~13–16 hours/month). Reserved strictly for high-throughput ARC Index multi-step agent benchmark sweeps.
- **Session Boundaries**:
  - Maximum continuous runtime cap: ~12 hours.
  - Idle timeout: 40–90 minutes (keep browser tab active; no headless background run without Ultra/Pro+).
  - Ephemeral `/content` storage: synced artifacts must persist externally.

---

### Dual Service Architecture:
1. **Daemon A (Port 8000)**: SGLang Server serving `cyankiwi/Qwen3.8-27B-AWQ-INT4` (NEXTN speculative decoding disabled: it does not fit the L4 alongside the Mamba cache).
2. **Daemon B (Port 8001)**: Gateway: Docling `POST /v1/parse_document`, plus `POST /v1/systemone` / `/v1/decisions`, which score options by next-token probability through SGLang `/v1/score` (no generated tokens).
3. **WAN Ingress**: Dual Cloudflare quick tunnels exporting public encrypted HTTPS endpoints for remote agents.
4. **Handoff**: cell [6] prints one command; `arc-cua doctor --pin` persists the gateway to `~/.omp/decision-endpoint.json`, which the local resolver health-checks on every run.

In [ ]:
# [1] Hardware Verification & Setup
import os, sys, subprocess, shutil

print("=== Checking NVIDIA L4 Hardware ===")
!nvidia-smi

print("\n=== Verifying Environment & Packages ===")
if not shutil.which("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb
    !rm -f cloudflared-linux-amd64.deb

# A fresh Colab runtime ships none of the serving stack; cell [2] imports
# sglang and cell [3] imports fastapi/uvicorn/docling. Install only what is missing.
import importlib.util
_missing = [pkg for mod, pkg in (("sglang", "sglang"), ("fastapi", "fastapi"),
            ("uvicorn", "uvicorn"), ("docling", "docling"), ("multipart", "python-multipart"))
            if importlib.util.find_spec(mod) is None]
if _missing:
    print("Installing:", " ".join(_missing))
    # Colab now runs Python 3.13, and the outlines_core pinned by sglang has no
    # cp313 wheel, so pip compiles it and needs a Rust toolchain on PATH.
    if not shutil.which("cargo"):
        subprocess.run("curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal",
                       shell=True, check=True)
        os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ["PATH"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", *_missing], check=True)

# Cell [2] waits ~90 s for SGLang; a cold 15 GB weight download takes far longer.
# Fetch the weights here so cell [2] only has to load them.
from huggingface_hub import snapshot_download
snapshot_download("cyankiwi/Qwen3.8-27B-AWQ-INT4")

print("cloudflared:", shutil.which("cloudflared"))
print("Environment ready for dual-service launch!")

In [ ]:
# [2] Daemon A: Launch SGLang Server with Qwen3.8-27B AWQ (Port 8000)
import subprocess, time, requests, os

MODEL_ID = "cyankiwi/Qwen3.8-27B-AWQ-INT4"
SGLANG_PORT = 8000

env = dict(os.environ)
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

sglang_cmd = [
    "python3", "-m", "sglang.launch_server",
    "--model-path", MODEL_ID,
    "--port", str(SGLANG_PORT),
    "--host", "127.0.0.1",
    "--mem-fraction-static", "0.96",
    "--max-running-requests", "1",
    "--context-length", "2048",
    "--attention-backend", "triton",
    "--disable-prefill-cuda-graph",
    "--disable-decode-cuda-graph"
]

print(f"Launching SGLang Server with {MODEL_ID} on port {SGLANG_PORT}...")
sglang_log = open("/content/sglang.log", "w")
sglang_proc = subprocess.Popen(sglang_cmd, stdout=sglang_log, stderr=subprocess.STDOUT, env=env)
print(f"SGLang PID: {sglang_proc.pid}. Weights cached locally; warming up server...")

ready = False
for i in range(150):  # up to ~300 s: 27B load + scheduler init exceeded 90 s on L4
    try:
        r = requests.get(f"http://127.0.0.1:{SGLANG_PORT}/v1/models", timeout=2)
        if r.status_code == 200:
            print(f"\n>>> SGLang Server is READY ({i*2}s)!")
            ready = True
            break
    except Exception:
        pass
    if sglang_proc.poll() is not None:
        print("\nSGLang exited prematurely. Log tail:")
        with open("/content/sglang.log") as f:
            print("".join(f.readlines()[-35:]))
        break
    time.sleep(2)
    if i % 5 == 0:
        print(f" [{i*2}s]", end="", flush=True)
    else:
        print(".", end="", flush=True)

if not ready and sglang_proc.poll() is None:
    print("\nServer still warming up. Check /content/sglang.log for status.")
# /v1/models answers before the Triton kernels are compiled; on L4 the first
# /v1/score call took ~280 s, and the gateway's 30 s upstream timeout turned every
# early decision into a 500. Pay that cost here, once, before cell [3] serves.
if ready:
    print("Warming up /v1/score (first call compiles kernels; can take ~5 min)...")
    _t0 = time.time()
    try:
        _w = requests.post(f"http://127.0.0.1:{SGLANG_PORT}/v1/score",
                           json={"model": MODEL_ID, "query": "Warm-up. Answer:", "items": [""],
                                 "label_token_ids": [9454, 2753], "apply_softmax": True},
                           timeout=600)
        print(f">>> /v1/score warm ({time.time() - _t0:.0f}s, HTTP {_w.status_code})")
    except Exception as e:
        print(f"Warm-up failed after {time.time() - _t0:.0f}s: {e}. Check /content/sglang.log.")


In [ ]:
# [3] Daemon B: Docling Extraction + SystemOne logit-decision gateway (Port 8001)
import subprocess, time, requests

GATEWAY_CODE = r'''
import os, tempfile, string, time, requests
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, Request
from typing import Optional

try:
    from docling.document_converter import DocumentConverter
    converter, init_err = DocumentConverter(), None
except Exception as e:
    converter, init_err = None, str(e)

app = FastAPI(title="Docling + SystemOne gateway")
SGLANG_URL = "http://127.0.0.1:8000"
MODEL_ID = "cyankiwi/Qwen3.8-27B-AWQ-INT4"
LETTERS = string.ascii_uppercase
_tok = {}

def token_id(s):
    if s not in _tok:
        ids = requests.post(f"{SGLANG_URL}/v1/tokenize", json={"model": MODEL_ID, "prompt": s, "add_special_tokens": False}, timeout=10).json()["tokens"]
        if len(ids) != 1:
            raise HTTPException(500, f"label {s!r} is not a single token")
        _tok[s] = ids[0]
    return _tok[s]

def wrap(user):
    # Qwen chat template with thinking disabled: the next token is the answer.
    return f"<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"

def next_token_dist(prompt, labels):
    r = requests.post(f"{SGLANG_URL}/v1/score", json={"model": MODEL_ID, "query": wrap(prompt), "items": [""],
        "label_token_ids": [token_id(l) for l in labels], "apply_softmax": True}, timeout=30)
    r.raise_for_status()
    body = r.json()
    return dict(zip(labels, body["scores"][0])), body.get("usage", {})

@app.get("/health")
def health():
    try:
        ok = requests.get(f"{SGLANG_URL}/v1/models", timeout=3).status_code == 200
    except Exception:
        ok = False
    return {"status": "ok", "docling_ready": converter is not None, "sglang_connected": ok}

@app.post("/v1/parse_document")
async def parse_document(file: UploadFile = File(None), file_path: Optional[str] = Form(None)):
    if converter is None:
        raise HTTPException(500, f"Docling unavailable: {init_err}")
    tmp = None
    try:
        if file is not None:
            tmp = tempfile.NamedTemporaryFile(delete=False, suffix=os.path.splitext(file.filename)[1] or ".pdf")
            tmp.write(await file.read()); tmp.close()
            path = tmp.name
        elif file_path and os.path.exists(file_path):
            path = file_path
        else:
            raise HTTPException(400, "file upload or valid file_path required")
        doc = converter.convert(path).document
        return {"status": "success", "markdown": doc.export_to_markdown(), "page_count": len(getattr(doc, "pages", []))}
    finally:
        if tmp is not None and os.path.exists(tmp.name):
            os.remove(tmp.name)

@app.post("/v1/systemone")
async def systemone(request: Request):
    data = await request.json()
    state = data.get("state", "")
    answers, prompt_tokens, t0 = {}, 0, time.perf_counter()
    for name, q in data.get("questions", {}).items():
        kind, ins = q.get("type", "choice"), q.get("instructions", "")
        if kind == "choice":
            opts = q.get("options", [])
            if not 1 <= len(opts) <= len(LETTERS):
                raise HTTPException(400, f"{name}: 1..{len(LETTERS)} options required, got {len(opts)}")
            labels = list(LETTERS[:len(opts)])
            listing = "\n".join(f"{l}. {o}" for l, o in zip(labels, opts))
            dist, u = next_token_dist(f"Context:\n{state}\n\nTask: {ins}\nOptions:\n{listing}\nAnswer with one letter.", labels)
            probs = {o: dist[l] for l, o in zip(labels, opts)}
            best = max(probs, key=probs.get)
            answers[name] = {"choice": best, "confidence": probs[best], "probabilities": probs}
        elif kind == "score":
            scale = [str(s) for s in q.get("scale", [1, 2, 3, 4, 5])]
            dist, u = next_token_dist(f"Context:\n{state}\n\nTask: {ins}\nAnswer with one number from {', '.join(scale)}.", scale)
            best = max(dist, key=dist.get)
            answers[name] = {"score": int(best), "expected": sum(int(k) * v for k, v in dist.items()), "confidence": dist[best], "probabilities": dist}
        elif kind in ("noul", "yes_no"):
            dist, u = next_token_dist(f"Context:\n{state}\n\nQuestion: {ins}\nAnswer yes or no.", ["yes", "no"])
            best = max(dist, key=dist.get)
            answers[name] = {"choice": best, "confidence": dist[best], "probabilities": dist}
        else:
            raise HTTPException(400, f"{name}: unknown type {kind!r}")
        prompt_tokens += u.get("prompt_tokens", 0)
    return {"answers": answers, "usage": {"prompt_tokens": prompt_tokens, "completion_tokens": 0},
            "latency_ms": (time.perf_counter() - t0) * 1000}

@app.post("/v1/decisions")
async def decisions(request: Request):
    return await systemone(request)

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=8001, log_level="warning")
'''

with open("/content/gateway_server.py", "w") as f:
    f.write(GATEWAY_CODE)

!pkill -9 -f gateway_server.py || true
time.sleep(1)
gw_proc = subprocess.Popen(["python3", "/content/gateway_server.py"], stdout=open("/content/gateway.log", "w"), stderr=subprocess.STDOUT)
for i in range(40):
    try:
        if requests.get("http://127.0.0.1:8001/health", timeout=1).status_code == 200:
            print(">>> Gateway READY"); break
    except Exception:
        pass
    time.sleep(1)
else:
    print(open("/content/gateway.log").read()[-2000:])

In [ ]:
# [4] Dual Cloudflare Quick Tunnels with Dynamic URL Extraction
import subprocess, re, time, json

def start_tunnel(port: int, name: str, log_file: str):
    cmd = ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"]
    log_f = open(log_file, "w")
    proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT)
    print(f"Started {name} tunnel (PID {proc.pid}). Searching for public HTTPS URL...")
    
    url = None
    url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    for _ in range(35):
        time.sleep(1)
        with open(log_file, "r") as f:
            content = f.read()
            matches = url_pattern.findall(content)
            if matches:
                url = matches[0]
                break
    return proc, url

sglang_proc, sglang_url = start_tunnel(8000, "SGLang Decision Server", "/content/tunnel_sglang.log")
docling_proc, docling_url = start_tunnel(8001, "Docling Extraction Server", "/content/tunnel_docling.log")

print("\n" + "="*70)
print("🚀 SYSTEM-ONE / SYSTEM-TWO DECISION BACKENDS ONLINE")
print("="*70)
print(f"▶ SGLang Base URL:      {sglang_url}")
print(f"  - Universal Decision: {sglang_url}/v1/decisions")
print(f"  - SystemOne (Jev):    {sglang_url}/v1/systemone")
print(f"  - Chat Completions:   {sglang_url}/v1/chat/completions")
print("-"*70)
print(f"▶ Docling Base URL:     {docling_url}")
print(f"  - Parse Document:     {docling_url}/v1/parse_document")
print(f"  - Health Status:      {docling_url}/health")
print("="*70)
# Machine-readable handoff: the local resolver reads this format.
with open("/content/endpoint.json", "w") as f:
    json.dump({"sglang_base_url": sglang_url, "gateway_base_url": docling_url,
               "systemone_url": f"{docling_url}/v1/systemone",
               "decisions_url": f"{docling_url}/v1/decisions"}, f, indent=2)

print("Set these URLs in your local client environment:")
print(f"export SGLANG_DECISION_ENDPOINT=\"{docling_url}/v1/systemone\"")
print(f"export SGLANG_BASE_URL=\"{sglang_url}\"")
print(f"export DOCLING_ENDPOINT=\"{docling_url}/v1/parse_document\"")

In [ ]:
# [5] End-to-End Smoke Test & Schema Validation
import requests, time

GATEWAY_URL = docling_url  # Port 8001 exposes /v1/systemone & /health
SGLANG_API = sglang_url   # Port 8000 exposes SGLang chat/completions

print(f"Testing SystemOne on Gateway Tunnel ({GATEWAY_URL})...")
test_payload = {
    "state": "Document: Invoice #INV-9021. Total Due: $450.00. Payment Terms: Net 30. Overdue.",
    "questions": {
        "target_folder": {
            "type": "choice",
            "instructions": "Route to the appropriate operational bucket.",
            "options": ["Finance/Invoices_and_Receipts", "Legal/Contracts", "Archive/Junk"]
        },
        "urgency": {
            "type": "score",
            "instructions": "Rate invoice urgency from 1 to 5.",
            "scale": [1, 2, 3, 4, 5]
        },
        "requires_action": {
            "type": "noul",
            "instructions": "Does this document require manual payment or approval?"
        }
    }
}
t0 = time.perf_counter()
try:
    res = requests.post(f"{GATEWAY_URL}/v1/systemone", json=test_payload, timeout=20)
    latency_ms = (time.perf_counter() - t0) * 1000
    print(f"Status: {res.status_code} ({latency_ms:.1f}ms)")
    print("Live Response from Qwen3.8-27B:", res.json())
except Exception as e:
    print("SystemOne Test Failed:", e)

print(f"\nTesting Docling /health on {GATEWAY_URL}...")
try:
    res = requests.get(f"{GATEWAY_URL}/health", timeout=5)
    print(f"Status: {res.status_code}, Response: {res.json()}")
except Exception as e:
    print("Docling Test Failed:", e)

print(f"\nTesting SGLang Models on {SGLANG_API}...")
try:
    res = requests.get(f"{SGLANG_API}/v1/models", timeout=5)
    print(f"Status: {res.status_code}, Models:", [m['id'] for m in res.json()['data']])
except Exception as e:
    print("SGLang Test Failed:", e)


In [ ]:
# [6] Optional: Stable Named Tunnel (arc.<your-domain>) via Cloudflare Access
# The quick tunnel in cell [4] reissues a random hostname every runtime start.
# A NAMED tunnel keeps one fixed hostname, so the local side stops changing too.
#
# PREREQUISITES (one-time, in the Cloudflare dashboard, not here):
#   1. Zero Trust > Networks > Tunnels > Create tunnel (cloudflared) -> "arc-sglang"
#   2. Zero Trust > Access > Applications > Add an application for
#      arc.<your-domain>, policy = "Service Auth" service token ONLY.
#   3. Public hostname arc.<your-domain> -> http://localhost:8001  (the GATEWAY,
#      not 8000; only the gateway serves /v1/systemone).
#   4. Access > Service Auth > Service tokens -> Create token. Put the TUNNEL token
#      in Colab secrets as ARC_TUNNEL_TOKEN, and the ACCESS token pair in your
#      laptop's environment as CF_ACCESS_CLIENT_ID / CF_ACCESS_CLIENT_SECRET.
#
# WHY THE TOKEN LIVES IN COLAB SECRETS, NOT IN THIS CELL: anyone who can view a
# shared or public notebook can read its source. A tunnel token is a credential
# that will happily connect to your machine. Colab Secrets keeps it out of the
# file, and the cell below only reads it at runtime.
#
# This cell is safe to run with no secret set: it falls back to the quick tunnel.

import os, re, json, subprocess, time

def _read_secret(name):
    """Read a Colab secret, tolerating a plain env var for local runs."""
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass
    return (os.environ.get(name) or "").strip()

_tunnel_token = _read_secret("ARC_TUNNEL_TOKEN")
NAMED_HOSTNAME = _read_secret("ARC_NAMED_HOSTNAME") or "arc.ihsanwanda.my.id"
USE_NAMED_TUNNEL = bool(_tunnel_token)

if USE_NAMED_TUNNEL:
    print(f"Starting NAMED tunnel for {NAMED_HOSTNAME} -> http://localhost:8001 ...")
    named_log_path = "/content/tunnel_named.log"
    # Truncate once, then let cloudflared hold the file for writing. Reopening it
    # per read on Windows-style locking is unreliable, and the cell must work on
    # a Colab Linux runtime as well as local dry-runs.
    with open(named_log_path, "w"):
        pass
    named_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--no-autoupdate", "run", "--token", _tunnel_token],
        stdout=open(named_log_path, "a"), stderr=subprocess.STDOUT,
    )
    # The connector must register with the edge before the hostname resolves.
    # Wait for its own readiness line rather than a fixed sleep.
    ready = False
    for _ in range(60):
        time.sleep(1)
        try:
            with open(named_log_path, errors="replace") as fh:
                content = fh.read()
        except OSError:
            content = ""
        if "Registered tunnel connection" in content or re.search(r"protocol=(quic|http2)", content):
            ready = True
            break
    if not ready:
        print("Named tunnel connector never reported registration in 60s.")
        try:
            with open(named_log_path, errors="replace") as fh:
                print(fh.read()[-1500:])
        except OSError:
            pass
    if ready:
        sglang_url = f"https://{NAMED_HOSTNAME}"
        docling_url = sglang_url
        print(">>> Named tunnel registered. The hostname below never changes.")
    else:
        print("Named tunnel did not register; falling back to the quick tunnel.")
        named_proc.terminate()
        USE_NAMED_TUNNEL = False

if not USE_NAMED_TUNNEL:
    print("No ARC_TUNNEL_TOKEN secret: using the quick-tunnel URLs from cell [4].")
    print(f"Gateway: {docling_url}")

# One handoff for both modes: the local `arc-cua doctor --pin` reads this file.
with open("/content/endpoint.json", "w") as f:
    json.dump({
        "mode": "named" if USE_NAMED_TUNNEL else "quick",
        "gateway_base_url": docling_url,
        "sglang_base_url": sglang_url,
        "systemone_url": f"{docling_url}/v1/systemone",
        "decisions_url": f"{docling_url}/v1/decisions",
        "access_required": bool(USE_NAMED_TUNNEL),
    }, f, indent=2)

print()
print("=" * 70)
if USE_NAMED_TUNNEL:
    print("STABLE URL. On your laptop, once:")
    print(f'  arc-cua doctor --endpoint "https://{NAMED_HOSTNAME}" --pin')
    print("The pin stays valid across Colab restarts, because the hostname is fixed.")
else:
    print("COPY THIS ONE LINE INTO YOUR LOCAL TERMINAL (URL changes each runtime):")
    print(f'  arc-cua doctor --endpoint "{docling_url}" --pin')
print("=" * 70)


In [ ]:
# [7] Hand Off To The Local Agent (no hand-typed exports)
# The quick tunnel issues a NEW hostname every runtime start, so this cell exists
# to collapse the handoff to a single copy-paste that pins the endpoint locally.
# On the machine running `arc-cua`, that command writes ~/.omp/decision-endpoint.json,
# and every later session (including this coding agent) resolves System-2 from it
# automatically -- no SGLANG_DECISION_ENDPOINT export required.
import json

print("=" * 70)
print("COPY THIS ONE LINE INTO YOUR LOCAL TERMINAL:")
print()
print(f'  arc-cua doctor --endpoint "{docling_url}" --pin')
print()
print("=" * 70)
print("Verify from the local agent with:  arc-cua doctor")
print("SGLang base URL (chat/scores only, NOT the bridge):", sglang_url)
print("Gateway base URL (serves /v1/systemone):", docling_url)

# Keep the kernel busy: Colab reclaims idle runtimes, and a dead gateway means
# every bridge call falls back to the rule heuristic.
print("\nColab session is live. Keep this tab open; re-run cells [2]-[4] if the GPU drops.")
